In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd
import random
from datetime import datetime
from PIL import Image
from tqdm import tqdm
import cv2
from torchvision import transforms
from torchvision.models.segmentation import deeplabv3_resnet101
import torch.nn as nn

# ============ SET RANDOM SEED ============
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

# ============ DEVICE ============
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# ============ LOAD YOUR DEEPLAB MODEL ============
CHECKPOINT_PATH = "deeplabv3_instrument.pt"

def build_deeplab_model():
    from torchvision.models.segmentation import deeplabv3_resnet101, DeepLabV3_ResNet101_Weights
    
    weights = DeepLabV3_ResNet101_Weights.DEFAULT
    model = deeplabv3_resnet101(weights=weights)
    model.classifier[4] = nn.Conv2d(256, 1, kernel_size=1)
    
    if model.aux_classifier is not None:
        model.aux_classifier[4] = nn.Conv2d(256, 1, kernel_size=1)
    
    return model

model = build_deeplab_model().to(device)

checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
if 'model_state_dict' in checkpoint:
    model.load_state_dict(checkpoint['model_state_dict'])
elif 'state_dict' in checkpoint:
    model.load_state_dict(checkpoint['state_dict'])
else:
    model.load_state_dict(checkpoint)

model.eval()
print(f"Model loaded successfully from {CHECKPOINT_PATH}")
print(f"  Val IoU:  {checkpoint.get('val_iou', 'N/A')}")
print(f"  Val Dice: {checkpoint.get('val_dice', 'N/A')}")

# ============ TRANSFORM ============
IMG_SIZE = 512
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# ============ PATHS ============
gt_folder = Path("data/gt")
labels_folder = Path("data/gt/labels_matched") #download the labels from the Endovis2018 dataset and place them in this folder

# ============ NOISE LEVELS TO RUN ============
NOISE_LEVELS = [50, 40, 30, 20, 10]
Gamma = "G3"

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_base = Path(f"results/downstream_tasks/synthetic/segmentation_results_{timestamp}")
output_base.mkdir(parents=True, exist_ok=True)

latest_base = Path("results/downstream_tasks/synthetic/segmentation_results_latest")
latest_base.mkdir(parents=True, exist_ok=True)

# ============ METHODS TEMPLATE ============
def get_methods(noise_level):
    """Get method folders for a specific noise level"""
    return {
        'Noisy': Path(f"data/synthetic_dataset_V/{Gamma}/{noise_level}"),
        'N2D': Path(f"results/sequential_eval/N2D/{Gamma}/{noise_level}"),
        'N2D_Colie': Path(f"results/sequential_eval/N2D_Colie/{Gamma}/{noise_level}"),
        '0Shot': Path(f"results/sequential_eval/0Shot/{Gamma}/{noise_level}"),
        '0Shot_Colie': Path(f"results/sequential_eval/0Shot_Colie/{Gamma}/{noise_level}"),
        'N2V': Path(f"results/sequential_eval/N2V/{Gamma}/{noise_level}"),
        'N2V_Colie': Path(f"results/sequential_eval/N2V_Colie/{Gamma}/{noise_level}"),
        'ZS-N2N_Pipeline': Path(f"results/Pipeline/ZSN2N_Pipeline/{Gamma}/{noise_level}"),
        'N2D_Pipeline': Path(f"results/Pipeline/N2D_Pipeline/{Gamma}/{noise_level}"),
    }

# ============ DRAW ZOOM HIGHLIGHT ============
def draw_zoom_highlight(ax, img_shape, zoom_center, zoom_size, color='red', linewidth=2):
    h, w = img_shape[:2]
    cx, cy = zoom_center
    half = zoom_size // 2
    
    x1 = max(0, cx - half)
    y1 = max(0, cy - half)
    x2 = min(w, cx + half)
    y2 = min(h, cy + half)
    
    rect = plt.Rectangle((x1, y1), x2-x1, y2-y1, 
                         fill=False, edgecolor=color, linewidth=linewidth)
    ax.add_patch(rect)

# ============ SEGMENT WITH DEEPLAB ============
def segment_with_deeplab(image_path, model, transform, img_size=IMG_SIZE):
    img = Image.open(image_path).convert("RGB")
    original_size = img.size
    
    input_tensor = transform(img).unsqueeze(0).to(device)
    
    with torch.no_grad():
        output = model(input_tensor)
        if isinstance(output, dict):
            output = output['out']
        elif hasattr(output, 'out'):
            output = output['out']
    
    logits = output.squeeze(1)
    pred_mask = (torch.sigmoid(logits) > 0.5).float().squeeze().cpu().numpy()
    
    mask_resized = Image.fromarray((pred_mask * 255).astype(np.uint8))
    mask_resized = mask_resized.resize(original_size, Image.NEAREST)
    mask_resized = np.array(mask_resized) > 127
    
    return mask_resized, np.array(img)

def get_gt_mask_from_label(label_path):
    if not label_path.exists():
        return None
    
    label_img = Image.open(label_path).convert("L")
    gt_mask = np.array(label_img)
    
    if gt_mask.max() > 1:
        gt_mask = (gt_mask > 0).astype(bool)
    else:
        gt_mask = gt_mask.astype(bool)
    
    return gt_mask

def get_label_path(labels_folder, img_name):
    
    label_path = labels_folder / f"{img_name}.bmp"
    if label_path.exists():
        return label_path
    
    label_path = labels_folder / f"{img_name}.png"
    if label_path.exists():
        return label_path
    
    return None

def compute_iou(mask_pred, mask_gt):
    if mask_pred is None or mask_gt is None:
        return 0.0
    
    if mask_pred.shape != mask_gt.shape:
        pred_img = Image.fromarray((mask_pred * 255).astype(np.uint8))
        pred_img = pred_img.resize((mask_gt.shape[1], mask_gt.shape[0]), Image.NEAREST)
        mask_pred = np.array(pred_img) > 127
    
    intersection = np.logical_and(mask_pred, mask_gt).sum()
    union = np.logical_or(mask_pred, mask_gt).sum()
    
    if union == 0:
        return 0.0
    return float(intersection) / float(union)

def compute_dice(mask_pred, mask_gt):
    if mask_pred is None or mask_gt is None:
        return 0.0
    
    if mask_pred.shape != mask_gt.shape:
        pred_img = Image.fromarray((mask_pred * 255).astype(np.uint8))
        pred_img = pred_img.resize((mask_gt.shape[1], mask_gt.shape[0]), Image.NEAREST)
        mask_pred = np.array(pred_img) > 127
    
    intersection = np.logical_and(mask_pred, mask_gt).sum()
    pred_sum = mask_pred.sum()
    gt_sum = mask_gt.sum()
    
    if pred_sum + gt_sum == 0:
        return 0.0
    return 2.0 * intersection / (pred_sum + gt_sum)

def create_overlay(image, mask):
    result = image.copy().astype(np.float32) / 255.0
    if mask is not None:
        color = np.array([0.0, 1.0, 0.5])
        for c in range(3):
            result[:, :, c] = result[:, :, c] * (1 - mask * 0.5) + color[c] * mask * 0.5
    return np.clip(result, 0, 1)

def get_gt_path(gt_folder, img_name):
    return gt_folder / img_name

# ============ MAIN FUNCTION ============
def compare_all_methods(noisy_files, labels_folder, methods, output_dir, latest_dir, model, transform, gt_folder):
    """Run DeepLab segmentation on all methods using labels as GT"""
    # --- SKIP LOGIC ---
    csv_path = output_dir / "segmentation_results.csv"
    completed = set()
    if csv_path.exists():
        try:
            existing_df = pd.read_csv(csv_path)
            completed = set(existing_df['filename'].tolist())
            print(f"  Found {len(completed)} already processed images")
        except:
            pass
    
    remaining_files = [f for f in noisy_files if f.name not in completed]
    if len(remaining_files) == 0:
        print("  All images already processed!")
        return [], {}, {}, {}
    
    print(f"  Processing {len(remaining_files)} remaining images")
    
    results = []
    global_intersection = {m: 0 for m in methods}
    global_union = {m: 0 for m in methods}
    global_dice = {m: 0.0 for m in methods}
    count_per_method = {m: 0 for m in methods}
    
    zoom_size = 256

    for img_path in tqdm(noisy_files, desc="Processing"):
        img_name = img_path.name
        label_path = get_label_path(labels_folder, img_name)
        
        if label_path is None:
            continue
        
        gt_mask = get_gt_mask_from_label(label_path)
        
        img_results = {'filename': img_name}
        overlays = {}
        originals = {}
        
        for method_name, folder_path in methods.items():
            method_path = folder_path / img_name
            
            if not method_path.exists():
                print(f"  {method_name}: image not found at {method_path}")
                continue
            
            pred_mask, img = segment_with_deeplab(method_path, model, transform)
            iou = compute_iou(pred_mask, gt_mask)
            dice = compute_dice(pred_mask, gt_mask)
            
            intersection = np.logical_and(pred_mask, gt_mask).sum()
            union = np.logical_or(pred_mask, gt_mask).sum()
            
            global_intersection[method_name] += intersection
            global_union[method_name] += union
            global_dice[method_name] += dice
            count_per_method[method_name] += 1
            
            img_results[f'iou_{method_name}'] = round(iou, 4)
            img_results[f'dice_{method_name}'] = round(dice, 4)
            originals[method_name] = img
            overlays[method_name] = create_overlay(img, pred_mask)
        
        n_rows = 1 + len(methods)
        fig, axes = plt.subplots(n_rows, 2, figsize=(10, n_rows * 3.5))
        if n_rows == 1:
            axes = axes.reshape(1, -1)
        
        gt_path = get_gt_path(gt_folder, img_name)
        if gt_path.exists():
            gt_img = np.array(Image.open(gt_path).convert("RGB"))
        else:
            gt_img = np.array(Image.open(label_path).convert("RGB"))
        
        gt_overlay = create_overlay(gt_img, gt_mask)
        zoom_center = (gt_img.shape[1] // 2, gt_img.shape[0] // 2)
        
        axes[0, 0].imshow(gt_img)
        draw_zoom_highlight(axes[0, 0], gt_img.shape, zoom_center, zoom_size, color='red')
        axes[0, 0].set_title("GT (Original + highlight)", fontsize=11)
        axes[0, 0].axis('off')
        
        axes[0, 1].imshow(gt_overlay)
        axes[0, 1].set_title("GT (Mask Overlay)", fontsize=11)
        axes[0, 1].axis('off')
        
        for idx, (method_name, _) in enumerate(methods.items(), start=1):
            if method_name in originals:
                iou_val = img_results.get(f'iou_{method_name}', 0.0)
                dice_val = img_results.get(f'dice_{method_name}', 0.0)
                
                axes[idx, 0].imshow(originals[method_name])
                draw_zoom_highlight(axes[idx, 0], originals[method_name].shape, zoom_center, zoom_size, color='red')
                axes[idx, 0].set_title(f"{method_name} (Original + highlight)", fontsize=11)
                axes[idx, 0].axis('off')
                
                axes[idx, 1].imshow(overlays[method_name])
                axes[idx, 1].set_title(f"{method_name} — IoU: {iou_val:.4f}, Dice: {dice_val:.4f}", fontsize=11)
                axes[idx, 1].axis('off')
            else:
                axes[idx, 0].axis('off')
                axes[idx, 1].axis('off')
        
        fig.suptitle(img_name, fontsize=13, fontweight='bold')
        plt.tight_layout()
        plt.savefig(output_dir / f"{img_name}", dpi=150, bbox_inches='tight')
        plt.savefig(latest_dir / f"{img_name}", dpi=150, bbox_inches='tight')
        plt.close()
        
        results.append(img_results)
    
    for method in methods:
        if count_per_method[method] > 0:
            global_dice[method] /= count_per_method[method]
    
    return results, global_intersection, global_union, global_dice

# ============ RUN ALL NOISE LEVELS ============
if __name__ == "__main__":
    all_results = {}
    
    for noise_level in NOISE_LEVELS:
        print("\n" + "=" * 80)
        print(f"PROCESSING NOISE LEVEL: {noise_level}")
        print("=" * 80)
        
        methods = get_methods(noise_level)
        noisy_folder = Path(f"data/synthetic_dataset_V/{Gamma}/{noise_level}")
        
        output_dir = output_base / f"noise_{noise_level}"
        output_dir.mkdir(parents=True, exist_ok=True)
        
        latest_dir = latest_base / f"noise_{noise_level}"
        latest_dir.mkdir(parents=True, exist_ok=True)
        
        noisy_files = sorted(noisy_folder.glob("*.png"))[:100]
        
        valid_files = []
        for f in noisy_files:
            label_path = get_label_path(labels_folder, f.name)
            if label_path is not None:
                valid_files.append(f)
        
        print(f"Found {len(valid_files)} images with matching labels")
        
        if len(valid_files) == 0:
            print(f"⚠️ No valid files for noise level {noise_level}, skipping...")
            continue
        
        results, global_intersection, global_union, global_dice = compare_all_methods(
            valid_files, labels_folder, methods, output_dir, latest_dir, model, transform, gt_folder
        )
        
        # Summary per noise level
        print("\n" + "-" * 80)
        print(f"SUMMARY — NOISE LEVEL {noise_level}")
        print("-" * 80)
        
        df = pd.DataFrame(results)
        all_results[noise_level] = df
        
        if not df.empty:
            print(f"{'Method':<25} {'Avg IoU':<15} {'Avg Dice':<15}")
            print("-" * 55)
            for method in methods.keys():
                iou_col = f'iou_{method}'
                dice_col = f'dice_{method}'
                if iou_col in df.columns:
                    avg_iou = df[iou_col].mean()
                    avg_dice = df[dice_col].mean()
                    print(f"{method:<25} {avg_iou:<15.4f} {avg_dice:<15.4f}")
        
        # Save per noise level CSV
        csv_path = output_dir / f"segmentation_results_noise_{noise_level}.csv"
        df.to_csv(csv_path, index=False)
        print(f"✅ Saved: {csv_path}")
    
    # ============ FINAL SUMMARY ACROSS ALL NOISE LEVELS ============
    print("\n" + "=" * 80)
    print("FINAL SUMMARY — ALL NOISE LEVELS")
    print("=" * 80)
    
    summary_data = []
    for noise_level, df in all_results.items():
        if not df.empty:
            row = {'noise_level': noise_level}
            for method in get_methods(noise_level).keys():
                iou_col = f'iou_{method}'
                if iou_col in df.columns:
                    row[f'iou_{method}'] = df[iou_col].mean()
                    row[f'dice_{method}'] = df[f'dice_{method}'].mean()
            summary_data.append(row)
    
    if summary_data:
        summary_df = pd.DataFrame(summary_data)
        summary_path = output_base / "summary_all_noise_levels.csv"
        summary_df.to_csv(summary_path, index=False)
        print(f"\n✅ Summary saved to: {summary_path}")
        
        print("\n--- Average IoU per Method across Noise Levels ---")
        print(f"{'Noise':<10}", end="")
        methods = get_methods(NOISE_LEVELS[0]).keys()
        for method in methods:
            print(f"{method:<20}", end="")
        print()
        print("-" * (10 + 20 * len(methods)))
        
        for _, row in summary_df.iterrows():
            print(f"{row['noise_level']:<10}", end="")
            for method in methods:
                val = row.get(f'iou_{method}', float('nan'))
                print(f"{val:<20.4f}", end="")
            print()
    
    print("\n" + "=" * 80)
    print("✅ ALL NOISE LEVELS COMPLETE")
    print("=" * 80)